# roberta
Original dataset annotations; validation selects the checkpoint.


## 1. Environment
On Kaggle, attach `vhngthnh/reconia-data`, enable Internet and a GPU, and enable the `GH_TOKEN` secret. Locally, use `data/raw/` or set `RECONIA_DATA_DIR` to the raw dataset root. Existing source and outputs are preserved.
Default `DATASET = "all"` runs `sroie`, `cord_v2`, then `wildreceipt` in one session. Set one dataset name to run only that dataset.

Source branch: `dev`. Existing checkouts are not updated automatically; update explicitly or start a fresh Kaggle session. Full training starts in section 4; run a GPU smoke check first.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

DATASET = "all"
GIT_REF = "dev"
repo_dir = Path("/kaggle/working/reconia") if Path("/kaggle").exists() else Path.cwd()
if repo_dir.name == "notebooks":
    repo_dir = repo_dir.parent
if not repo_dir.exists():
    import base64
    from kaggle_secrets import UserSecretsClient
    env = os.environ.copy()
    token = UserSecretsClient().get_secret("GH_TOKEN")
    auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
               GIT_CONFIG_VALUE_0="Authorization: Basic " + auth)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF,
                    "https://github.com/tamir39/reconia.git", str(repo_dir)], env=env, check=True)
    del env, token, auth
source_branch = subprocess.check_output(
    ["git", "-C", str(repo_dir), "branch", "--show-current"], text=True
).strip()
if source_branch != GIT_REF:
    raise RuntimeError(
        f"Expected branch {GIT_REF}, found {source_branch or 'detached HEAD'}. "
        "Use a fresh Kaggle session or update the checkout explicitly before running."
    )
source_commit = subprocess.check_output(
    ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True
).strip()
print("Source:", source_branch, source_commit)
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)


## 2. Prepare and inspect the selected dataset


In [ ]:
from importlib import import_module
from utils.common import get_cache_dir, get_data_dir

datasets = ("sroie", "cord_v2", "wildreceipt") if DATASET == "all" else (DATASET,)
for dataset in datasets:
    print("Raw data:", get_data_dir(dataset))
    prepare = getattr(import_module(f"prepare.prepare_{dataset}"), f"prepare_{dataset}")
    prepare()
    print("Prepared:", get_cache_dir(dataset))


In [ ]:
preview_dataset = datasets[0]
import json
from IPython.display import Image, display

with (get_cache_dir(preview_dataset) / "train.jsonl").open(encoding="utf-8") as file:
    sample = json.loads(file.readline())
image_path = get_data_dir(preview_dataset) / sample["image_path"]
print("Sample:", sample["id"], "| Image:", image_path)
display(Image(filename=str(image_path), width=350))
display({"text_box_excerpt": sample["ocr"][:5], "target_gt": sample["gt"]})


## 3. Pretrained weights


In [ ]:
from tools.download_weights import download_weights

download_weights("roberta")


## 4. Train and validate
Precision, Recall and F1 use exact field-value matches after lowercasing and whitespace normalization, averaged across fields (Macro). Validation runs after each epoch; its Macro-F1 selects the best checkpoint.


In [ ]:
from models.roberta.train import train_model

for dataset in datasets:
    train_model(dataset, epochs=30, batch_size=8, accum_steps=1,
                lr=3e-5, min_epochs=5, patience=5, resume=False)


## 5. Validation history


In [ ]:
import pandas as pd

for dataset in datasets:
    print(dataset)
    display(pd.read_csv(repo_dir / "results" / dataset / "roberta" / "history.csv"))


## 6. Final test (run after experiments are fixed)
Loads the best validation checkpoint; does not update weights.


In [ ]:
RUN_TEST = False
if RUN_TEST:
    for dataset in datasets:
        train_model(dataset, batch_size=8, test_only=True)
